[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/tokenization-embeddings-course/01_unicode_bytes/01_proyecto_unicode_bytes.ipynb)

# Proyecto 01 · Auditoría Unicode del corpus Políglota + normalizador
**Módulo 01 · Texto como bytes: Unicode, UTF-8, normalización**

| | |
|---|---|
| **Nivel** | 🟢 Básico |
| **Duración estimada** | ≈ 1.5 h |
| **GPU recomendada** | Ninguna (CPU) |
| **Entregable** | funciones `audit_doc`, `normalizar` y el informe `poliglota_unicode.json` |

## Contexto de negocio
Políglota Labs va a ingerir documentación de muchas fuentes (wikis, PDFs convertidos, emails, páginas web). Tu CTO ha visto cosas raras en las pruebas: búsquedas que no encuentran «canción» aunque está en el documento, palabras que parecen iguales y no lo son, texto con «Ã³» en vez de «ó»… Te pide **una auditoría Unicode del corpus** y **un normalizador** que lo deje en una forma canónica *antes* de tokenizar y de indexar.

Como el corpus real (Wikipedia) está bastante limpio, usaremos un **inyector de ruido con verdad conocida** (`contaminar`): sabemos exactamente qué documentos ensuciamos y de qué manera, así que podemos medir **precisión y recall de tus detectores** y **qué fracción del daño repara tu normalizador**.

**Conexión.** *Anterior (módulo 00):* usas `carga_corpus`, las métricas y el BM25 que construiste. *Siguiente (módulo 02):* tu `normalizar` será el primer paso antes de entrenar el BPE propio sobre el corpus; la calidad de la normalización determina el vocabulario de fusiones.

## Dataset
Corpus Políglota (`carga_corpus`, módulo 00): Wikipedia ES/EN + código Python, con cadena **oficial → espejo → sintético** (la celda de carga imprime la fuente). El inyector solo contamina documentos ES/EN (el código tiene reglas propias).

## Entregables y criterios de evaluación (rúbrica)

| # | Criterio | Umbral |
|---|---|---|
| R1 | `escritura` / `palabra_mixta` pasan sus tests (homoglifos latín/cirílico/griego, sin falsos positivos en palabras monoescritura) | tests ✅ |
| R2 | `parece_mojibake` y `arregla_mojibake` pasan sus tests (incluido el **doble mojibake** y el caso de falso positivo «é + NBSP») | tests ✅ |
| R3 | `audit_doc` devuelve las 12 claves y cuenta correctamente en los casos de prueba (emoji de familia, NFD, invisibles) | tests ✅ |
| R4 | `normalizar` pasa sus tests (idempotente; conserva el ZWJ de los emojis; quita invisibles/bidi dentro de palabras; arregla homoglifos y ancho completo) | tests ✅ |
| R5 | **Detección** por tipo de ruido sobre el corpus contaminado | recall ≥ **0.95** y precisión ≥ **0.90** en cada tipo; documentos limpios marcados ≤ **2 %** |
| R6 | **Reparación**: `normalizar(sucio) == normalizar(original)` en los documentos contaminados; idempotencia en todo el corpus; documentos limpios que cambian más que NFC | recuperación ≥ **0.95**; idempotencia = 100 %; limpios sin cambios ≥ **0.99** |
| R7 | **Efecto en búsqueda** (BM25 con tokenizer léxico sin normalizar, consultas ES/EN): nDCG@10 limpio vs sucio vs normalizado | la contaminación baja el nDCG ≥ **0.03** y el corpus normalizado queda a ≤ **0.01** del limpio |
| R8 | **Efecto en coste de tokens** (GPT-2, 150 docs): tokens(sucio) / tokens(normalizado) | ≥ **1.02** (si `tiktoken` no está disponible, se omite) |
| R9 | `poliglota_unicode.json` guardado con las claves requeridas | ✅ |

Umbrales calibrados con **lo que mide la solución de referencia** (corpus **sintético**, `FAST_DEV_RUN=True`, semilla 42, `tasa=0.3`): 93 documentos contaminados; recall = precisión = **1.00** en los 7 tipos; recuperación **93/93**; limpios sin cambios **400/400**; nDCG@10 limpio **0.955**, sucio **0.894**, normalizado **0.955**; tokens sucio/normalizado **1.049**. Los umbrales llevan margen. Con Wikipedia **real** no pudimos medir la solución en el entorno de autoría (Hugging Face bloqueado): es esperable que la precisión baje, porque Wikipedia contiene de forma natural *ZWNJ*, NBSP, mezclas de escrituras legítimas, etc.; en ese caso **inspecciona los «falsos positivos»**: suelen ser anomalías reales, no errores tuyos.

## 0. Preparación (celdas dadas)

In [ ]:
!pip install -q numpy pandas matplotlib regex tiktoken rank_bm25

In [ ]:
import json, math, pathlib, random, re, sys, time, unicodedata, collections
import numpy as np, pandas as pd, regex
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED)
FAST_DEV_RUN = True
CHECKS: dict = {}

def chequea(nombre, fn):
    """Ejecuta fn() y registra ✅/❌/⏳ sin romper «Ejecutar todo»."""
    try:
        fn(); CHECKS[nombre] = True; print("✅", nombre)
    except NotImplementedError as e:
        CHECKS[nombre] = None; print("⏳ pendiente:", nombre, "→", e)
    except AssertionError as e:
        CHECKS[nombre] = False; print("❌", nombre, "→", e)
    except Exception as e:  # noqa: BLE001
        CHECKS[nombre] = False; print("💥", nombre, "→", type(e).__name__, e)

In [ ]:
# ── carga_corpus (1/3) ──
# ── Datos para el corpus SINTÉTICO de Políglota (último recurso si no hay red / Hugging Face) ──
# (artículo+concepto ES, artículo+concepto EN, definición ES, definición EN)
CONCEPTOS = [
    ("el índice invertido", "the inverted index", "una estructura que asocia cada término con la lista de documentos que lo contienen", "a structure that maps each term to the list of documents that contain it"),
    ("la caché", "the cache", "una memoria rápida que guarda resultados recientes para no recalcularlos", "a fast memory that stores recent results so they are not recomputed"),
    ("la base de datos relacional", "the relational database", "un sistema que organiza los datos en tablas relacionadas mediante claves", "a system that organizes data into tables related through keys"),
    ("el balanceador de carga", "the load balancer", "un componente que reparte las peticiones entre varios servidores", "a component that spreads requests across several servers"),
    ("el contenedor Docker", "the Docker container", "un proceso aislado que empaqueta una aplicación con sus dependencias", "an isolated process that packages an application with its dependencies"),
    ("el control de versiones", "version control", "un sistema que registra los cambios de los archivos a lo largo del tiempo", "a system that records changes to files over time"),
    ("la API REST", "the REST API", "una interfaz que expone recursos mediante URL y verbos HTTP", "an interface that exposes resources through URLs and HTTP verbs"),
    ("la autenticación con tokens", "token authentication", "un mecanismo que identifica al usuario con un testigo firmado en cada petición", "a mechanism that identifies the user with a signed token on every request"),
    ("la expresión regular", "the regular expression", "un patrón que describe un conjunto de cadenas de texto", "a pattern that describes a set of text strings"),
    ("el hilo de ejecución", "the thread", "una secuencia de instrucciones que comparte memoria con otros hilos del proceso", "a sequence of instructions that shares memory with other threads of the process"),
    ("el recolector de basura", "the garbage collector", "un mecanismo que libera automáticamente la memoria que ya no se usa", "a mechanism that automatically frees memory that is no longer used"),
    ("la cola de mensajes", "the message queue", "un búfer que desacopla a los productores de los consumidores de eventos", "a buffer that decouples event producers from consumers"),
    ("la tabla hash", "the hash table", "una estructura que guarda pares clave-valor y busca en tiempo casi constante", "a structure that stores key-value pairs and looks them up in near-constant time"),
    ("el árbol binario de búsqueda", "the binary search tree", "un árbol ordenado donde cada nodo separa los valores menores de los mayores", "an ordered tree where each node separates smaller values from larger ones"),
    ("la red neuronal", "the neural network", "un modelo formado por capas de unidades que aprenden pesos a partir de ejemplos", "a model made of layers of units that learn weights from examples"),
    ("el descenso de gradiente", "gradient descent", "un método que ajusta los parámetros en la dirección que reduce la pérdida", "a method that adjusts parameters in the direction that reduces the loss"),
    ("el sobreajuste", "overfitting", "el fenómeno por el que un modelo memoriza el entrenamiento y generaliza mal", "the phenomenon where a model memorizes the training set and generalizes poorly"),
    ("la validación cruzada", "cross-validation", "una técnica que estima el rendimiento repartiendo los datos en varios pliegues", "a technique that estimates performance by splitting the data into several folds"),
    ("el tokenizador", "the tokenizer", "el componente que convierte el texto en una secuencia de identificadores enteros", "the component that turns text into a sequence of integer identifiers"),
    ("la ventana de contexto", "the context window", "la cantidad máxima de tokens que un modelo puede leer a la vez", "the maximum number of tokens a model can read at once"),
    ("el compilador", "the compiler", "un programa que traduce código fuente a instrucciones de máquina", "a program that translates source code into machine instructions"),
    ("el sistema de archivos", "the file system", "la capa que organiza cómo se guardan y se nombran los datos en disco", "the layer that organizes how data is stored and named on disk"),
    ("la memoria virtual", "virtual memory", "una abstracción que da a cada proceso su propio espacio de direcciones", "an abstraction that gives each process its own address space"),
    ("el protocolo TCP", "the TCP protocol", "un protocolo que entrega bytes de forma ordenada y fiable entre dos máquinas", "a protocol that delivers bytes reliably and in order between two machines"),
    ("el cifrado simétrico", "symmetric encryption", "un esquema que usa la misma clave para cifrar y descifrar", "a scheme that uses the same key to encrypt and decrypt"),
    ("las pruebas unitarias", "unit tests", "comprobaciones automáticas que verifican una función aislada", "automatic checks that verify a single isolated function"),
    ("la integración continua", "continuous integration", "una práctica que compila y prueba cada cambio de forma automática", "a practice that builds and tests every change automatically"),
    ("los microservicios", "microservices", "una arquitectura que divide la aplicación en servicios pequeños e independientes", "an architecture that splits the application into small independent services"),
    ("la paginación", "pagination", "una técnica que devuelve los resultados en bloques de tamaño fijo", "a technique that returns results in fixed-size blocks"),
    ("la limitación de tasa", "rate limiting", "una política que restringe cuántas peticiones puede hacer un cliente por minuto", "a policy that restricts how many requests a client can make per minute"),
    ("la serialización JSON", "JSON serialization", "el proceso de convertir objetos en texto JSON y viceversa", "the process of converting objects to JSON text and back"),
    ("la variable de entorno", "the environment variable", "un par nombre-valor que el sistema operativo entrega a cada proceso", "a name-value pair that the operating system hands to each process"),
    ("el entorno virtual", "the virtual environment", "una carpeta aislada con su propio intérprete de Python y sus paquetes", "an isolated folder with its own Python interpreter and packages"),
    ("el decorador", "the decorator", "una función que envuelve a otra para añadirle comportamiento", "a function that wraps another one to add behavior"),
    ("el generador", "the generator", "una función que produce valores perezosamente con la palabra yield", "a function that lazily produces values with the yield keyword"),
    ("el manejo de excepciones", "exception handling", "el mecanismo para capturar errores sin que el programa se detenga", "the mechanism for catching errors without stopping the program"),
    ("la programación asíncrona", "asynchronous programming", "un modelo que solapa esperas de entrada y salida sin bloquear el hilo", "a model that overlaps input and output waits without blocking the thread"),
    ("el bloqueo de base de datos", "the database lock", "un candado que impide que dos transacciones modifiquen la misma fila a la vez", "a lock that prevents two transactions from modifying the same row at once"),
    ("la migración de esquema", "the schema migration", "un script versionado que cambia la estructura de la base de datos", "a versioned script that changes the structure of the database"),
    ("el registro de eventos", "event logging", "la práctica de escribir mensajes con marca de tiempo sobre lo que hace el sistema", "the practice of writing timestamped messages about what the system does"),
]
HERRAMIENTAS = ["PostgreSQL", "Redis", "nginx", "Docker", "pytest", "Kubernetes", "FastAPI", "SQLite", "Git", "Linux", "Celery", "Prometheus"]
PARAMETROS = ["max_connections", "timeout_seconds", "cache_size", "batch_size", "log_level", "retry_count", "pool_size", "num_workers"]
# aspecto -> (título ES, título EN, palabra clave ES, palabra clave EN, plantillas ES, plantillas EN)
ASPECTOS = {
    "intro": ("{n}: qué es y para qué sirve", "{n}: what it is and what it is for", "qué es", "what is",
        ["Se usa en {t} para simplificar el trabajo diario del equipo.", "Entender {c} es un requisito antes de pasar a producción.", "Su idea central es sencilla y aparece en casi todos los sistemas modernos.", "La documentación de {t} dedica un capítulo completo a {c}."],
        ["It is used in {t} to simplify the daily work of the team.", "Understanding {c} is a prerequisite before going to production.", "Its core idea is simple and appears in almost every modern system.", "The {t} documentation devotes a whole chapter to {c}."]),
    "config": ("{n}: cómo configurarlo", "{n}: how to configure it", "configurar", "configure",
        ["Para configurar {c} edita el archivo de ajustes y reinicia {t}.", "El parámetro {p} controla el comportamiento de {c} y su valor por defecto es {x}.", "Después de cambiar {p} a {x}, comprueba el resultado con un comando de diagnóstico.", "Guarda la configuración de {c} en un archivo versionado y no en la consola."],
        ["To configure {c}, edit the settings file and restart {t}.", "The {p} parameter controls the behavior of {c} and its default value is {x}.", "After changing {p} to {x}, check the result with a diagnostic command.", "Keep the configuration of {c} in a versioned file and not in the console."]),
    "errores": ("{n}: errores frecuentes", "{n}: common errors", "errores", "errors",
        ["Un error frecuente con {c} es olvidar el valor de {p} y dejarlo en {x}.", "Si aparece un fallo intermitente en {t}, revisa primero los registros de {c}.", "Muchos equipos descubren el problema de {c} solo cuando la carga se multiplica por {x}.", "El mensaje de error suele ser engañoso: la causa real está en {c}, no en {t}."],
        ["A common mistake with {c} is forgetting {p} and leaving it at {x}.", "If an intermittent failure shows up in {t}, check the logs of {c} first.", "Many teams only discover the problem with {c} when the load grows by {x} times.", "The error message is usually misleading: the real cause is in {c}, not in {t}."]),
    "rendimiento": ("{n}: rendimiento y optimización", "{n}: performance and optimization", "rendimiento", "performance",
        ["Para medir el rendimiento de {c} usa un perfilador y no tu intuición.", "Subir {p} a {x} mejora la latencia, pero consume más memoria en {t}.", "La optimización prematura de {c} suele complicar el código sin ganar tiempo.", "En una prueba con {x} usuarios, {c} redujo la latencia mediana a la mitad."],
        ["To measure the performance of {c}, use a profiler and not your intuition.", "Raising {p} to {x} improves latency but uses more memory in {t}.", "Premature optimization of {c} usually complicates the code without saving time.", "In a test with {x} users, {c} cut the median latency in half."]),
    "practicas": ("{n}: buenas prácticas", "{n}: best practices", "buenas prácticas", "best practices",
        ["Una buena práctica con {c} es documentar {p} y automatizar su despliegue en {t}.", "Revisa {c} en cada revisión de código y escribe pruebas para los casos límite.", "Mantén {c} simple: añade complejidad solo cuando los datos lo justifiquen.", "Define alertas sobre {c} antes del primer incidente, no después."],
        ["A good practice with {c} is to document {p} and automate its deployment in {t}.", "Review {c} in every code review and write tests for the edge cases.", "Keep {c} simple: add complexity only when the data justifies it.", "Define alerts on {c} before the first incident, not after."]),
    "ejemplo": ("{n}: ejemplo en Python", "{n}: Python example", "ejemplo", "example",
        ["En este ejemplo en Python usamos {t} para mostrar {c} paso a paso.", "El script define {p} = {x} y luego imprime el resultado de {c}.", "Ejecuta el ejemplo con Python 3.11 y compara la salida con la esperada.", "Puedes adaptar el código de {c} a tu propio proyecto cambiando solo {p}."],
        ["In this Python example we use {t} to show {c} step by step.", "The script defines {p} = {x} and then prints the result of {c}.", "Run the example with Python 3.11 and compare the output with the expected one.", "You can adapt the code for {c} to your own project by changing only {p}."]),
    "seguridad": ("{n}: consideraciones de seguridad", "{n}: security considerations", "seguridad", "security",
        ["Desde el punto de vista de la seguridad, {c} nunca debe exponerse sin autenticación.", "Rota las credenciales de {t} cada {x} días y no las guardes en el repositorio.", "Un ataque contra {c} suele aprovechar el valor de {p} mal configurado.", "Audita los accesos a {c} y conserva los registros durante {x} días."],
        ["From a security standpoint, {c} must never be exposed without authentication.", "Rotate the {t} credentials every {x} days and do not store them in the repository.", "An attack on {c} usually exploits a misconfigured {p} value.", "Audit access to {c} and keep the logs for {x} days."]),
}

In [ ]:
# ── carga_corpus (2/3) ──
# ── Generador del corpus sintético (mismo esquema que el real) ────────────────
import random


def _es_fix(s: str) -> str:
    """Contracciones del español: 'a el' -> 'al', 'de el' -> 'del'."""
    return s.replace(" a el ", " al ").replace(" de el ", " del ").replace("De el ", "Del ").replace("A el ", "Al ")


def _sintetico_wiki(lang: str, n_docs: int, n_queries: int, seed: int = 42):
    """Docs técnicos sintéticos ES/EN: concepto x aspecto x herramienta. Devuelve (docs, queries, qrels)."""
    rng = random.Random(seed + (0 if lang == "es" else 1))
    li = 0 if lang == "es" else 1
    combos = [(c, a, t) for c in range(len(CONCEPTOS)) for a in ASPECTOS for t in range(len(HERRAMIENTAS))]
    rng.shuffle(combos)
    docs, titles = [], []
    for k, (ci, asp, ti) in enumerate(combos[:n_docs]):
        concepto, definicion = CONCEPTOS[ci][li], CONCEPTOS[ci][2 + li]
        tool = HERRAMIENTAS[ti]
        tit_t, plantillas = ASPECTOS[asp][li], ASPECTOS[asp][4 + li]
        nombre = concepto.split(" ", 1)[1] if concepto.split(" ", 1)[0] in ("el", "la", "los", "las", "the") else concepto
        slots = lambda: dict(c=concepto, t=tool, p=rng.choice(PARAMETROS), x=rng.choice([8, 16, 30, 64, 100, 256, 1024]))
        frases = [("{C} " + ("es " if li == 0 else "is ") + definicion + ".").format(C=concepto)]
        frases += [pl.format(**slots()) for pl in rng.sample(plantillas, len(plantillas))]
        frases.append(rng.choice(ASPECTOS["intro"][4 + li]).format(**slots()))
        for _ in range(2):  # ruido: frases de otro concepto (hace el retrieval no trivial)
            otro = rng.randrange(len(CONCEPTOS))
            otra = rng.choice(list(ASPECTOS.values()))[4 + li]
            s = slots(); s["c"] = CONCEPTOS[otro][li]
            frases.append(rng.choice(otra).format(**s))
        s = slots()
        frases.append(("Ejemplo: `{p} = {x}`." if li == 0 else "Example: `{p} = {x}`.").format(**s))
        text = " ".join(f[0].upper() + f[1:] for f in frases)
        docs.append({"id": f"{lang}-syn-{k:04d}", "lang": lang, "text": _es_fix(text) if lang == "es" else text})
        titles.append((nombre, ASPECTOS[asp][2 + li], tool, tit_t.format(n=nombre) + f" ({tool})"))
    queries, qrels = [], {}
    for j, k in enumerate(rng.sample(range(len(docs)), min(n_queries, len(docs)))):
        nombre, kw, tool, titulo = titles[k]
        q = titulo.replace(":", "").replace("(", "").replace(")", "") if j % 2 == 0 else f"{nombre} {kw} {tool}"
        qid = f"{lang}-q{j:04d}"
        queries.append({"qid": qid, "lang": lang, "text": q})
        qrels[qid] = {docs[k]["id"]: 1}
    return docs, queries, qrels

In [ ]:
# ── carga_corpus (3/3) ──
# ── carga_corpus(): cadena oficial (HF) -> espejo (API de Wikipedia / stdlib) -> sintético ─────
import ast
import json
import os
import random
import urllib.parse
import urllib.request

SEED = 42


def _strip_docstring(func_src: str):
    """Devuelve (código sin docstring, primera frase del docstring) o None si no hay docstring útil."""
    try:
        fn = ast.parse(func_src).body[0]
        doc = ast.get_docstring(fn)
        first = fn.body[0]
    except Exception:
        return None
    if not doc or not (isinstance(first, ast.Expr) and isinstance(getattr(first, "value", None), ast.Constant)):
        return None
    lines = func_src.splitlines()
    code = "\n".join(lines[: first.lineno - 1] + lines[first.end_lineno :])
    frase = doc.strip().split("\n\n")[0].replace("\n", " ").strip()
    frase = frase.split(". ")[0].rstrip(".")
    return (code, frase) if len(frase.split()) >= 4 else None


def _code_stdlib(n_docs: int, n_queries: int, rng: random.Random):
    """Espejo OFFLINE con código real: funciones de la biblioteca estándar con su docstring como query."""
    import importlib, inspect, textwrap
    mods = ["argparse", "bisect", "calendar", "collections", "colorsys", "copy", "csv", "difflib", "fnmatch", "fractions",
            "functools", "getpass", "glob", "heapq", "html", "json", "operator", "os", "pathlib", "posixpath", "queue",
            "random", "re", "shlex", "shutil", "statistics", "string", "tempfile", "textwrap", "urllib.parse", "uuid", "zipfile"]
    cands = []
    for name in mods:
        try:
            src = inspect.getsource(importlib.import_module(name))
            tree = ast.parse(src)
        except Exception:
            continue
        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.end_lineno - node.lineno >= 4:
                seg = textwrap.dedent("\n".join(src.splitlines()[node.lineno - 1 : node.end_lineno]))
                r = _strip_docstring(seg)
                if r and 200 <= len(r[0]) <= 2500:
                    cands.append((f"{name}.{node.name}", r[0], r[1]))
    rng.shuffle(cands)
    return cands[:n_docs]


def _code_hf(n_docs: int, rng: random.Random):
    """Oficial: CodeSearchNet (Python) en streaming. Prueba varios repos por cambios de formato en `datasets`."""
    from datasets import load_dataset
    errs = []
    for repo, kw in [("code-search-net/code_search_net", {"name": "python"}), ("code_search_net", {"name": "python"}),
                     ("claudios/code_search_net", {"name": "python"}), ("Nan-Do/code-search-net-python", {})]:
        try:
            ds = load_dataset(repo, split="train", streaming=True, **kw).shuffle(seed=SEED, buffer_size=2000)
            out = []
            for ex in ds:
                src = ex.get("func_code_string") or ex.get("whole_func_string") or ex.get("code") or ""
                r = _strip_docstring(src) if '"""' in src or "'''" in src else None
                doc = (ex.get("func_documentation_string") or ex.get("docstring") or "").strip().split("\n\n")[0]
                doc = doc.replace("\n", " ").split(". ")[0].rstrip(".")
                code = r[0] if r else src
                if 200 <= len(code) <= 2500 and len(doc.split()) >= 4:
                    out.append((ex.get("func_name") or f"csn.{len(out)}", code, doc))
                if len(out) >= n_docs:
                    return out
        except Exception as e:  # noqa: BLE001
            errs.append(f"{repo}: {type(e).__name__}")
    raise RuntimeError("CodeSearchNet no disponible -> " + "; ".join(errs))


def _wiki_hf(lang: str, n_docs: int, cap: int):
    """Oficial: wikimedia/wikipedia (snapshot 20231101) en streaming con un buffer pequeño."""
    from datasets import load_dataset
    ds = load_dataset("wikimedia/wikipedia", f"20231101.{lang}", split="train", streaming=True)
    ds = ds.shuffle(seed=SEED, buffer_size=2000)
    out = []
    for ex in ds:
        if len(ex["text"]) >= 600 and len(ex["title"]) >= 3:
            out.append((f"{lang}-{ex['id']}", ex["title"], ex["text"][:cap]))
        if len(out) >= n_docs:
            break
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out


def _wiki_api(lang: str, n_docs: int):
    """Espejo: API pública de Wikipedia (artículos aleatorios, solo la sección introductoria; ~20 por petición)."""
    out, seen, tries = [], set(), 0
    while len(out) < n_docs and tries < 80:
        tries += 1
        qs = urllib.parse.urlencode({"action": "query", "format": "json", "generator": "random", "grnnamespace": 0,
                                     "grnlimit": 20, "prop": "extracts", "exintro": 1, "explaintext": 1, "exlimit": 20})
        req = urllib.request.Request(f"https://{lang}.wikipedia.org/w/api.php?{qs}",
                                     headers={"User-Agent": "poliglota-course/1.0 (curso educativo)"})
        with urllib.request.urlopen(req, timeout=20) as r:
            pages = json.load(r).get("query", {}).get("pages", {})
        for p in pages.values():
            if p["pageid"] not in seen and len(p.get("extract", "")) >= 300:
                seen.add(p["pageid"])
                out.append((f"{lang}-{p['pageid']}", p["title"], p["extract"]))
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out[:n_docs]


def carga_corpus(fast_dev: bool = True) -> dict:
    """Corpus Políglota -> {"docs": [{id, lang, text}], "queries": [{qid, lang, text}], "qrels": {qid: {doc_id: 1}},
    "source": "real" | "synthetic"}. Cadena: oficial (HF) -> espejo (API Wikipedia; código de la stdlib) -> sintético."""
    rng = random.Random(SEED)
    n_wiki, n_code, q_wiki, q_code, cap = (150, 100, 40, 30, 1500) if fast_dev else (3000, 1500, 200, 150, 6000)
    docs, queries, qrels, source = [], [], {}, "real"
    wiki = {}
    for lang in ("es", "en"):
        for nombre, fn in (("HF wikimedia/wikipedia", lambda: _wiki_hf(lang, n_wiki, cap)),
                           ("API de Wikipedia (espejo)", lambda: _wiki_api(lang, min(n_wiki, 400)))):
            try:
                wiki[lang] = fn()
                print(f"[carga_corpus] Wikipedia {lang}: {len(wiki[lang])} artículos desde {nombre}")
                break
            except Exception as e:  # noqa: BLE001
                print(f"[carga_corpus] {nombre} ({lang}) falló: {type(e).__name__}: {str(e)[:80]}")
    if len(wiki) == 2:
        for lang, arts in wiki.items():
            docs += [{"id": i, "lang": lang, "text": t} for i, _, t in arts]
            for j, k in enumerate(rng.sample(range(len(arts)), min(q_wiki, len(arts)))):
                qid = f"{lang}-q{j:04d}"
                queries.append({"qid": qid, "lang": lang, "text": arts[k][1]})
                qrels[qid] = {arts[k][0]: 1}
    else:
        source = "synthetic"
        print("⚠️  [carga_corpus] SIN acceso a Wikipedia: uso texto ES/EN SINTÉTICO (mismo esquema). Los números NO son comparables con datos reales.")
        for lang in ("es", "en"):
            d, q, r = _sintetico_wiki(lang, n_wiki, q_wiki, SEED)
            docs += d; queries += q; qrels.update(r)
    try:
        code = _code_hf(n_code, rng)
        print(f"[carga_corpus] código: {len(code)} funciones desde CodeSearchNet (HF)")
    except Exception as e:  # noqa: BLE001
        print(f"[carga_corpus] CodeSearchNet falló ({str(e)[:90]}); uso funciones REALES de la stdlib de Python (espejo offline)")
        code = _code_stdlib(n_code, q_code, rng)
    for k, (name, src, doc) in enumerate(code):
        docs.append({"id": f"code-{k:04d}", "lang": "code", "text": src})
    for j, k in enumerate(rng.sample(range(len(code)), min(q_code, len(code)))):
        qid = f"code-q{j:04d}"
        queries.append({"qid": qid, "lang": "code", "text": code[k][2]})
        qrels[qid] = {f"code-{k:04d}": 1}
    return {"docs": docs, "queries": queries, "qrels": qrels, "source": source}

In [ ]:
# ── Métricas canónicas ──
# ── Métricas de retrieval canónicas de Políglota (módulo 00) ──────────────────
import math

import numpy as np


def _dedup(ranked_ids):
    """Quita duplicados conservando la primera aparición (un doc repetido NO puntúa dos veces)."""
    seen, out = set(), []
    for d in ranked_ids:
        if d not in seen:
            seen.add(d)
            out.append(d)
    return out


def dcg_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """DCG con ganancia lineal (como trec_eval / pytrec_eval, que usa BEIR/MTEB) y descuento 1/log2(rango+1)."""
    ranked = _dedup(ranked_ids)[:k]
    return sum(rels.get(d, 0) / math.log2(i + 2) for i, d in enumerate(ranked) if rels.get(d, 0) > 0)


def ndcg_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """nDCG@k. rels: doc_id -> relevancia (>0 relevante). Sin relevantes -> 0.0 (evaluate() las excluye)."""
    ideal = sorted((r for r in rels.values() if r > 0), reverse=True)[:k]
    idcg = sum(r / math.log2(i + 2) for i, r in enumerate(ideal))
    return 0.0 if idcg == 0 else dcg_at_k(ranked_ids, rels, k) / idcg


def recall_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """Fracción de los documentos relevantes que aparecen en el top-k."""
    relevant = {d for d, r in rels.items() if r > 0}
    if not relevant:
        return 0.0
    return len(relevant & set(_dedup(ranked_ids)[:k])) / len(relevant)


def mrr_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """1/rango del primer relevante dentro del top-k (0 si no hay ninguno)."""
    for i, d in enumerate(_dedup(ranked_ids)[:k]):
        if rels.get(d, 0) > 0:
            return 1.0 / (i + 1)
    return 0.0


def rank_from_scores(scores, doc_ids, k: int | None = None):
    """Ordena por score desc. Empates -> orden original del corpus (determinista). Los empates son una fuente
    silenciosa de varianza: con scores iguales, el orden de entrada decidiría el resultado."""
    scores = np.asarray(scores, dtype=float)
    order = np.lexsort((np.arange(len(scores)), -scores))
    order = order[:k] if k else order
    return [doc_ids[i] for i in order]


def evaluate(run: dict, qrels: dict, k: int = 10) -> dict:
    """run: {qid: [doc_id ordenados]}. Promedia solo sobre queries con algún relevante (convención trec_eval)."""
    n, tot = 0, {"ndcg": 0.0, "recall": 0.0, "mrr": 0.0}
    for qid, rels in qrels.items():
        if not any(r > 0 for r in rels.values()):
            continue
        ranked = run.get(qid, [])
        tot["ndcg"] += ndcg_at_k(ranked, rels, k)
        tot["recall"] += recall_at_k(ranked, rels, k)
        tot["mrr"] += mrr_at_k(ranked, rels, k)
        n += 1
    out = {m: (v / n if n else 0.0) for m, v in tot.items()}
    out.update(n_queries=n, n_skipped=len(qrels) - n)
    return out

In [ ]:
# ── BM25 (módulo 00) ──
# ── Tokenización simple + BM25 propio (módulo 00) ─────────────────────────────
import math
import re
import unicodedata
from collections import Counter, defaultdict

import numpy as np

_WORD_RE = re.compile(r"\w+", re.UNICODE)


def fold_accents(text: str) -> str:
    """'canción' -> 'cancion' (NFD y descartar marcas combinantes Mn)."""
    nfd = unicodedata.normalize("NFD", text)
    return unicodedata.normalize("NFC", "".join(c for c in nfd if unicodedata.category(c) != "Mn"))


def tokenize_simple(text: str, accents: str = "keep") -> list[str]:
    """Minúsculas + palabras \\w+. accents='fold' quita tildes. (Esto NO es el 'token' de un LLM: es un token léxico.)"""
    text = unicodedata.normalize("NFC", text).lower()
    if accents == "fold":
        text = fold_accents(text)
    return _WORD_RE.findall(text)


class BM25:
    """BM25 con índice invertido. idf='lucene': ln(1+(N-n+.5)/(n+.5)) (siempre >= 0).
    idf='okapi': ln((N-n+.5)/(n+.5)) con suelo epsilon*idf_medio (como rank_bm25.BM25Okapi)."""

    def __init__(self, k1: float = 1.2, b: float = 0.75, idf: str = "lucene", epsilon: float = 0.25):
        self.k1, self.b, self.idf_kind, self.epsilon = k1, b, idf, epsilon

    def fit(self, docs_tokens: list[list[str]], doc_ids: list[str]) -> "BM25":
        self.doc_ids = list(doc_ids)
        self.N = len(docs_tokens)
        self.dl = np.array([len(t) for t in docs_tokens], dtype=float)
        self.avgdl = float(self.dl.mean()) if self.N else 0.0
        self.postings = defaultdict(list)  # término -> [(índice_doc, tf)]
        for i, toks in enumerate(docs_tokens):
            for term, tf in Counter(toks).items():
                self.postings[term].append((i, tf))
        raw = {t: len(p) for t, p in self.postings.items()}
        if self.idf_kind == "lucene":
            self.idf = {t: math.log(1 + (self.N - n + 0.5) / (n + 0.5)) for t, n in raw.items()}
        else:
            self.idf = {t: math.log(self.N - n + 0.5) - math.log(n + 0.5) for t, n in raw.items()}
            floor = self.epsilon * sum(self.idf.values()) / len(self.idf)
            self.idf = {t: (v if v >= 0 else floor) for t, v in self.idf.items()}
        return self

    def scores(self, q_tokens: list[str]) -> np.ndarray:
        s = np.zeros(self.N)
        for term in q_tokens:  # los términos repetidos de la query suman varias veces (igual que rank_bm25)
            for i, tf in self.postings.get(term, ()):
                norm = self.k1 * (1 - self.b + self.b * self.dl[i] / self.avgdl)
                s[i] += self.idf[term] * tf * (self.k1 + 1) / (tf + norm)
        return s

    def search(self, q_tokens: list[str], k: int = 10) -> list[str]:
        s = self.scores(q_tokens)
        order = np.lexsort((np.arange(self.N), -s))[:k]  # empates -> orden del corpus
        return [self.doc_ids[i] for i in order]

In [ ]:
import hashlib
import os
import tempfile
import urllib.request


def get_gpt2_encoding():
    """tiktoken 'gpt2'. 1) oficial (openaipublic); 2) espejo en GitHub (graykode/gpt-2-Pytorch) verificado con los
    mismos SHA-256 que exige tiktoken; 3) None (las celdas que lo usan se omiten con un aviso)."""
    import tiktoken
    try:
        return tiktoken.get_encoding("gpt2")
    except Exception as e:  # noqa: BLE001
        print(f"[tiktoken] descarga oficial no disponible ({type(e).__name__}); pruebo el espejo de GitHub...")
    try:
        import tiktoken_ext.openai_public as op
        from tiktoken.load import data_gym_to_mergeable_bpe_ranks
        base = "https://raw.githubusercontent.com/graykode/gpt-2-Pytorch/master/GPT2/"
        sha = {"vocab.bpe": "1ce1664773c50f3e0cc8842619a93edc4624525b728b188a9e0be33b7726adc5",
               "encoder.json": "196139668be63f3b5d6574427317ae82f612a97c5d1cdaf36ed2256dbf636783"}
        paths = {}
        for name, h in sha.items():
            data = urllib.request.urlopen(base + name, timeout=60).read()
            assert hashlib.sha256(data).hexdigest() == h, f"hash de {name} no coincide"
            paths[name] = os.path.join(tempfile.gettempdir(), "gpt2_" + name)
            with open(paths[name], "wb") as f:
                f.write(data)
        ranks = data_gym_to_mergeable_bpe_ranks(vocab_bpe_file=paths["vocab.bpe"], encoder_json_file=paths["encoder.json"])
        print("[tiktoken] GPT-2 cargado desde el espejo de GitHub (hashes verificados)")
        return tiktoken.Encoding("gpt2", pat_str=op.r50k_pat_str, mergeable_ranks=ranks,
                                 special_tokens={"<|endoftext|>": 50256})
    except Exception as e:  # noqa: BLE001
        print(f"[tiktoken] sin GPT-2 ({type(e).__name__}: {str(e)[:80]}). Las comparaciones con tiktoken se omiten.")
        return None

In [ ]:
# ── Dado: inyector de ruido con verdad conocida ──
ZW = ["\u200b", "\u2060", "\ufeff"]
BIDI = ["\u202e", "\u202d", "\u2066", "\u2069"]
HOMOGLIFOS = {"a": "\u0430", "e": "\u0435", "o": "\u043e", "p": "\u0440", "c": "\u0441", "x": "\u0445", "i": "\u0456"}  # letras cirílicas
TIPOS = ["nfd", "zero_width", "homoglifo", "mojibake", "nbsp", "ancho_completo", "bidi"]

def _p_nfd(t, r): return unicodedata.normalize("NFD", t)
def _p_zw(t, r):
    ws = list(re.finditer(r"[^\W\d_]{4,}", t)); r.shuffle(ws)
    for m in sorted(ws[:5], key=lambda m: -m.start()):
        k = m.start() + r.randint(1, len(m.group()) - 1); t = t[:k] + r.choice(ZW) + t[k:]
    return t
def _p_homoglifo(t, r):
    ws = [m for m in re.finditer(r"[a-z]{4,}", t) if any(ch in HOMOGLIFOS for ch in m.group())]; r.shuffle(ws)
    for m in sorted(ws[:5], key=lambda m: -m.start()):
        w = m.group(); k = r.choice([i for i, ch in enumerate(w) if ch in HOMOGLIFOS])
        t = t[:m.start()] + w[:k] + HOMOGLIFOS[w[k]] + w[k + 1:] + t[m.end():]
    return t
def _p_mojibake(t, r): return t.encode("utf-8").decode("latin-1")
def _p_nbsp(t, r): return re.sub(r"(?<=\w) (?=\w)", lambda m: "\u00a0" if r.random() < 0.3 else " ", t)
def _p_ancho(t, r): return "".join(chr(ord(c) + 0xFEE0) if "!" <= c <= "~" and r.random() < 0.2 else c for c in t)
def _p_bidi(t, r):
    k = r.randrange(len(t)); return t[:k] + r.choice(BIDI) + t[k:]
_PERT = dict(nfd=_p_nfd, zero_width=_p_zw, homoglifo=_p_homoglifo, mojibake=_p_mojibake, nbsp=_p_nbsp, ancho_completo=_p_ancho, bidi=_p_bidi)

def contaminar(docs, tasa=0.3, seed=42):
    """Copia de `docs` con UNA perturbación en ≈ `tasa` de los documentos (no de código).
    Devuelve (docs_sucios, verdad) con verdad = {doc_id: tipo} solo si el texto realmente cambió."""
    r = random.Random(seed); sucios, verdad = [], {}
    for d in docs:
        d2 = dict(d)
        if d["lang"] != "code" and r.random() < tasa:
            tipos = TIPOS[:]; r.shuffle(tipos)                      # el primer tipo que realmente cambie el texto
            for tipo in tipos:
                nuevo = _PERT[tipo](d["text"], r)
                if nuevo != d["text"]:
                    d2["text"] = nuevo; verdad[d["id"]] = tipo; break
        sucios.append(d2)
    return sucios, verdad


def evaluar_deteccion(audits, verdad, detectores):
    """Precisión y recall por tipo. `audits`: {doc_id: dict de audit_doc}; `detectores`: {tipo: f(audit) -> bool}."""
    filas = {}
    for tipo, f in detectores.items():
        tp = sum(1 for i, t in verdad.items() if t == tipo and f(audits[i]))
        fn = sum(1 for i, t in verdad.items() if t == tipo and not f(audits[i]))
        fp = sum(1 for i, a in audits.items() if f(a) and i not in verdad)       # marcados pero LIMPIOS (sin inyección)
        filas[tipo] = {"verdaderos": tp + fn, "recall": tp / max(tp + fn, 1), "FP en limpios": fp,
                       "precisión (vs limpios)": tp / max(tp + fp, 1)}
    return pd.DataFrame(filas).T

In [ ]:
corpus = carga_corpus(fast_dev=FAST_DEV_RUN)
docs, queries, qrels, source = corpus["docs"], corpus["queries"], corpus["qrels"], corpus["source"]
sucios, verdad = contaminar(docs, tasa=0.3, seed=SEED)
orig = {d["id"]: d for d in docs}
print(f"source = {source!r} | {len(docs)} docs | {len(verdad)} contaminados:", dict(collections.Counter(verdad.values())))
ejemplo = next(i for i, t in verdad.items() if t == "homoglifo")
a, b = orig[ejemplo]["text"], next(d for d in sucios if d["id"] == ejemplo)["text"]
print("\nEjemplo de homoglifo →", [(x, y) for x, y in zip(a.split(), b.split()) if x != y][:2], "(¡se ven iguales!)")

## TODO 1 · Escrituras y palabras mixtas
Implementa `escritura(ch)` (la «escritura» aproximada de un carácter: la **primera palabra de su nombre Unicode**, `LATIN`, `CYRILLIC`, `GREEK`…; `"?"` si no tiene nombre) y `palabra_mixta(w)`: `True` si las **letras** de la palabra pertenecen a **≥ 2** escrituras distintas entre `{LATIN, CYRILLIC, GREEK}`.

<details>
<summary>🪜 Pista 1</summary>

`unicodedata.name(ch, '')` devuelve p. ej. `'CYRILLIC SMALL LETTER A'`. Toma la primera palabra con `.split()[0]`.

</details>

<details>
<summary>🪜 Pista 2</summary>

Construye el conjunto `{escritura(c) for c in w if c.isalpha()}` y comprueba que su intersección con `{'LATIN','CYRILLIC','GREEK'}` tiene al menos 2 elementos.

</details>

In [ ]:
def escritura(ch):
    # TODO
    raise NotImplementedError("TODO 1a: implementa escritura")

def palabra_mixta(w):
    # TODO
    raise NotImplementedError("TODO 1b: implementa palabra_mixta")

def test_mixta():
    assert escritura("a") == "LATIN" and escritura("а") == "CYRILLIC" and escritura("ο") == "GREEK"
    assert not palabra_mixta("paypal") and palabra_mixta("pаypal") and palabra_mixta("aοb")
    assert not palabra_mixta("привет") and not palabra_mixta("ñandú") and not palabra_mixta("123")
chequea("R1 palabra_mixta", test_mixta)

## TODO 2 · Mojibake: detectar y reparar
`parece_mojibake(t)` debe ser `True` cuando el texto contiene secuencias de **UTF-8 leído como Latin-1/cp1252** (`Ã³`, `Ã±`, `Â¿`, `â€`…) y **`False`** en texto sano, incluido el caso engañoso «é + NBSP» (`"qué es"`: `é`=U+00E9 y NBSP=U+00A0 están en los rangos de «mojibake» pero **no** forman UTF-8 válido). `arregla_mojibake(t)` revierte el error (`encode('latin-1')` o `'cp1252'` + `decode('utf-8')`); debe poder con el **doble mojibake** y dejar intacto el texto sano.

<details>
<summary>🪜 Pista 1</summary>

Un patrón candidato: un carácter en `U+00C2…U+00F4` seguido de 1–3 caracteres en `U+0080…U+00BF`. Pero un candidato **solo cuenta** si al pasarlo por `.encode('latin-1').decode('utf-8')` no lanza error (prueba prefijos del candidato, de largo a corto).

</details>

<details>
<summary>🪜 Pista 2</summary>

`arregla_mojibake`: repite hasta 3 veces: si `parece_mojibake`, prueba `for enc in ('latin-1', 'cp1252')` con `t.encode(enc).decode('utf-8')` dentro de un `try/except UnicodeError` y quédate con el primer resultado distinto. Añade la firma `'â€'` (comillas/guiones en cp1252) a la detección.

</details>

In [ ]:
def parece_mojibake(t):
    # TODO
    raise NotImplementedError("TODO 2a: implementa parece_mojibake")

def arregla_mojibake(t):
    # TODO
    raise NotImplementedError("TODO 2b: implementa arregla_mojibake")

def test_mojibake():
    assert parece_mojibake("CanciÃ³n") and parece_mojibake("Â¿quÃ©?") and not parece_mojibake("canción ¿qué?")
    assert not parece_mojibake("qué es"), "falso positivo: é + NBSP no es mojibake (no es UTF-8 válido)"
    assert arregla_mojibake("CanciÃ³n") == "Canción" and arregla_mojibake("hola") == "hola"
    doble = "Canción".encode().decode("latin-1").encode().decode("latin-1")
    assert arregla_mojibake(doble) == "Canción", "doble mojibake: necesita 2 pasadas"
    assert arregla_mojibake("“hola".encode().decode("cp1252")) == "“hola"
chequea("R2 mojibake", test_mojibake)

## TODO 3 · Auditoría de un documento
Implementa `audit_doc(t)` → `dict` con estas 12 claves: `chars` (code points), `graphemes` (con `regex` y `\X`), `bytes` (UTF-8), `no_nfc` (¿el texto **no** está en NFC?), `zero_width` (nº de caracteres de `ZW_CHARS`), `bidi` (nº de `BIDI_CHARS`), `nbsp` (nº de separadores `Zs` distintos del espacio normal), `ancho_completo` (nº de caracteres `U+FF01…U+FF5E`), `mixed_words` (nº de palabras `\w+` mixtas), `mojibake` (bool), `control` (nº de caracteres de categoría `Cc` salvo `\n`, `\t`, `\r`) y `fffd` (nº de `U+FFFD`).

<details>
<summary>🪜 Pista 1</summary>

Casi todo son cuentas de caracteres con `sum(c in CONJUNTO for c in t)` o con `unicodedata.category(c)`. Las palabras salen de `regex.findall(r'\w+', t)`.

</details>

<details>
<summary>🪜 Pista 2</summary>

`no_nfc = t != unicodedata.normalize('NFC', t)`; `graphemes = len(regex.findall(r'\X', t))`; `nbsp = sum(unicodedata.category(c) == 'Zs' and c != ' ' for c in t)`.

</details>

In [ ]:
ZW_CHARS = set("​‌‍⁠﻿")                     # ZWSP, ZWNJ, ZWJ, WORD JOINER, BOM
BIDI_CHARS = set("‪‫‬‭‮⁦⁧⁨⁩")   # embeddings/overrides/isolates (Trojan Source)

def audit_doc(t):
    # TODO: devuelve el dict con las 12 claves
    raise NotImplementedError("TODO 3: implementa audit_doc")

CLAVES_AUDIT = {"chars", "graphemes", "bytes", "no_nfc", "zero_width", "bidi", "nbsp", "ancho_completo", "mixed_words", "mojibake", "control", "fffd"}
def test_audit():
    a = audit_doc("👨‍👩‍👧‍👦")
    assert set(a) == CLAVES_AUDIT, f"claves: {set(a) ^ CLAVES_AUDIT}"
    assert (a["chars"], a["graphemes"], a["bytes"], a["zero_width"]) == (7, 1, 25, 3)
    assert audit_doc("é")["no_nfc"] and not audit_doc("é")["no_nfc"]
    assert audit_doc("a​b")["zero_width"] == 1 and audit_doc("a‮b")["bidi"] == 1 and audit_doc("a b")["nbsp"] == 1
    assert audit_doc("ＡＢｃ")["ancho_completo"] == 3 and audit_doc("pаypal")["mixed_words"] == 1 and audit_doc("CanciÃ³n")["mojibake"]
chequea("R3 audit_doc", test_audit)

## TODO 4 · El normalizador
Implementa `normalizar(t, nfkc=False)`. Reglas (en este orden):

1. **Arregla mojibake** (`arregla_mojibake`).
2. **NFC** (o NFKC si `nfkc=True`).
3. **Pliega el ancho completo** (`U+FF01…U+FF5E` → ASCII: `chr(ord(c) - 0xFEE0)`) — mejor que NFKC para todo: es un martillo menos grande.
4. **Quita** los caracteres de `ZW_QUITAR` (`U+200B`, `U+2060`, `U+FEFF`) y los de `BIDI_CHARS`.
5. Quita **ZWJ/ZWNJ** (`U+200D`, `U+200C`) **solo** cuando están entre dos alfanuméricos **ASCII** (los de los emojis y de las escrituras índicas/persa se **conservan**).
6. Convierte cualquier separador `Zs` (NBSP, espacio fino, ideográfico…) en un espacio normal `" "`, y elimina controles `Cc` salvo `\n \t \r`.
7. En cada palabra **mixta** (`palabra_mixta`), sustituye los homoglifos por su letra latina con el diccionario `CONFUSABLES`.

Debe ser **idempotente** (`normalizar(normalizar(x)) == normalizar(x)`) y dejar **intacto** el texto sano salvo la NFC.

<details>
<summary>🪜 Pista 1</summary>

Encadena transformaciones simples sobre `t`; para el paso 5 usa `re.sub(r'(?<=[A-Za-z0-9])[\u200c\u200d](?=[A-Za-z0-9])', '', t)`; para el 7, `regex.sub(r'\w+', f, t)` con una función `f` que reciba la coincidencia.

</details>

<details>
<summary>🪜 Pista 2</summary>

Paso 7: `lambda m: ''.join(CONFUSABLES.get(c, c) for c in m.group()) if palabra_mixta(m.group()) else m.group()`. Paso 6: `''.join(' ' if unicodedata.category(c) == 'Zs' else c for c in t)`.

</details>

In [ ]:
ZW_QUITAR = set("​⁠﻿")
CONFUSABLES = {"а": "a", "е": "e", "о": "o", "р": "p", "с": "c", "х": "x", "і": "i",
               "ο": "o", "α": "a", "ѕ": "s", "ј": "j"}

def plegar_ancho(t):
    # TODO: U+FF01..U+FF5E -> ASCII
    raise NotImplementedError("TODO 4a: implementa plegar_ancho")

def normalizar(t, nfkc=False):
    # TODO: aplica las 7 reglas en orden
    raise NotImplementedError("TODO 4b: implementa normalizar")

def test_normalizar():
    assert normalizar("é") == "é" and normalizar("１２３ ＡＢＣ") == "123 ABC"
    assert normalizar("ho​la mundo") == "hola mundo" and normalizar("a‮b") == "ab"
    assert normalizar("pаypal") == "paypal" and normalizar("привет") == "привет"
    assert normalizar("👨‍👩‍👧") == "👨‍👩‍👧", "el ZWJ de los emojis se conserva"
    assert normalizar("ab‍cd") == "abcd", "ZWJ entre letras ASCII se elimina"
    assert normalizar("CanciÃ³n") == "Canción"
    for t in ("é​ ＡＢ", "pаypal x", "CanciÃ³n Â¿", "hola mundo"):
        assert normalizar(normalizar(t)) == normalizar(t), f"no idempotente en {t!r}"
    assert normalizar("½ ² ﬁ") == "½ ² ﬁ", "sin nfkc, la compatibilidad se conserva"
chequea("R4 normalizar", test_normalizar)

## TODO 5 · ¿Detecta tu auditoría lo que inyectamos?
Rellena `DETECTORES`: un `dict` `tipo → función(audit) → bool` para los 7 tipos (`nfd`, `zero_width`, `homoglifo`, `mojibake`, `nbsp`, `ancho_completo`, `bidi`) usando solo las claves de `audit_doc`. La función dada `evaluar_deteccion` calcula **recall** (¿detectas los contaminados?) y **precisión frente a los limpios** (¿marcas documentos sin inyección?).

<details>
<summary>🪜 Pista 1</summary>

Cada detector es una condición sobre una clave: p. ej. `nfd` ⇒ `a['no_nfc']`; `zero_width` ⇒ `a['zero_width'] > 0`.

</details>

<details>
<summary>🪜 Pista 2</summary>

`homoglifo` ⇒ `a['mixed_words'] > 0`; `ancho_completo` ⇒ `a['ancho_completo'] > 0`; `mojibake` ⇒ `a['mojibake']`.

</details>

In [ ]:
DETECTORES = {}   # TODO: {"nfd": lambda a: ..., ...} con los 7 tipos

try:
    audits = {d["id"]: audit_doc(d["text"]) for d in sucios}
    assert set(DETECTORES) == set(TIPOS), f"faltan detectores: {set(TIPOS) - set(DETECTORES)}"
    det = evaluar_deteccion(audits, verdad, DETECTORES)
    display(det.round(3))
except NotImplementedError as e:
    print("⏳ pendiente:", e)
except AssertionError as e:
    print("⏳", e)

## TODO 6 · ¿Repara el normalizador?
Calcula (a) la **tasa de recuperación**: fracción de documentos contaminados cuyo `normalizar(sucio)` es **igual** a `normalizar(original)`; (b) la **idempotencia** en todo el corpus sucio; (c) la fracción de documentos **limpios** que `normalizar` deja igual que su NFC. Guárdalos en `REPARACION = {"recuperacion": ..., "idempotente": bool, "limpios_intactos": ...}`.

<details>
<summary>🪜 Pista 1</summary>

Itera sobre `sucios` y compara con `orig[d['id']]['text']`. Para los limpios usa `docs` (el corpus sin contaminar).

</details>

<details>
<summary>🪜 Pista 2</summary>

`recuperacion = mean(normalizar(d['text']) == normalizar(orig[d['id']]['text']) for d in sucios if d['id'] in verdad)`; idempotente con `all(...)`; limpios con `unicodedata.normalize('NFC', t)`.

</details>

In [ ]:
REPARACION = {}
def calcula_reparacion():
    # TODO: devuelve {"recuperacion": float, "idempotente": bool, "limpios_intactos": float}
    raise NotImplementedError("TODO 6: implementa calcula_reparacion")

try:
    REPARACION = calcula_reparacion(); print(REPARACION)
except NotImplementedError as e:
    print("⏳ pendiente:", e)

## TODO 7 · Impacto en búsqueda y en coste de tokens
**(a) Búsqueda.** Con el BM25 del módulo 00 y un tokenizer léxico **sin** normalizar (`naive`), calcula el nDCG@10 (consultas ES/EN) sobre tres versiones del corpus: `docs` (limpio), `sucios` y `sucios` normalizados. Rellena `NDCG = {"limpio":…, "sucio":…, "normalizado":…}`.
**(b) Tokens.** Con GPT-2 (`get_gpt2_encoding()`), cuenta los tokens de los primeros 150 documentos no-código de `sucios` **antes y después** de `normalizar`, y guarda `TOKENS = {"sucio": …, "normalizado": …, "ratio": …}`.

<details>
<summary>🪜 Pista 1</summary>

Reutiliza `BM25`, `evaluate` y `qrels`. Indexa solo documentos y consultas que no sean `code`. El tokenizer léxico es `re.findall(r'\w+', t.lower())`.

</details>

<details>
<summary>🪜 Pista 2</summary>

`variante = [dict(d, text=normalizar(d['text'])) for d in sucios]`; después una función `ndcg_variante(docs_v)` que construya el índice, busque las consultas y llame a `evaluate(...)['ndcg']`.

</details>

In [ ]:
naive = lambda t: re.findall(r"\w+", t.lower())          # tokenizer léxico SIN normalizar: expone el efecto

def ndcg_variante(docs_v):
    # TODO: BM25 sobre docs_v (sin código) + consultas ES/EN -> nDCG@10
    raise NotImplementedError("TODO 7a: implementa ndcg_variante")

NDCG, TOKENS = {}, {}
try:
    NDCG = {"limpio": ndcg_variante(docs), "sucio": ndcg_variante(sucios),
            "normalizado": ndcg_variante([dict(d, text=normalizar(d["text"])) for d in sucios])}
    print({k: round(v, 4) for k, v in NDCG.items()})
    fig, ax = plt.subplots(figsize=(5.5, 3.4)); ax.bar(list(NDCG), list(NDCG.values()), color=["#2CA02C", "#D62728", "#1F6FEB"])
    ax.set_ylim(0, 1.05); ax.set_ylabel("nDCG@10"); ax.set_title(f"Ruido Unicode y normalización (corpus {source})"); plt.tight_layout(); plt.show()
except NotImplementedError as e:
    print("⏳ pendiente:", e)

enc = get_gpt2_encoding()
try:
    if enc is None:
        print("tiktoken no disponible: R8 se omite")
    else:
        muestra = [d for d in sucios if d["lang"] != "code"][:150]
        ts = sum(len(enc.encode(d["text"])) for d in muestra); tn = sum(len(enc.encode(normalizar(d["text"]))) for d in muestra)
        TOKENS = {"sucio": ts, "normalizado": tn, "ratio": ts / tn}; print(TOKENS)
except NotImplementedError as e:
    print("⏳ pendiente:", e)

> 👀 **Qué debes observar.** Tres barras: **limpio** (verde) y **normalizado** (azul) deben quedar prácticamente a la misma altura, y **sucio** (rojo) varios puntos por debajo (la solución mide 0.955 / 0.894 / 0.955 con corpus sintético). Si la azul no alcanza a la verde, tu `normalizar` deja algún tipo de ruido sin reparar: vuelve a la tabla de TODO 5 y mira qué tipo tiene recall bajo. En la salida de tokens, un ratio > 1 es el sobrecoste en tokens que el ruido Unicode añade a la factura de un LLM.

## TODO 8 · Informe
Guarda `poliglota_unicode.json` con las claves `source`, `n_docs`, `n_contaminados`, `deteccion` (la tabla de TODO 5 como dict), `reparacion`, `ndcg`, `tokens`, `bytes_por_char` (por idioma, del corpus **limpio**, usando tu `audit_doc`) y `version_unicode` (`unicodedata.unidata_version`).

In [ ]:
CLAVES_INFORME = ["source", "n_docs", "n_contaminados", "deteccion", "reparacion", "ndcg", "tokens", "bytes_por_char", "version_unicode"]

def guardar_informe(path="poliglota_unicode.json"):
    # TODO: construye el dict con CLAVES_INFORME, guárdalo como JSON y devuélvelo
    raise NotImplementedError("TODO 8: implementa guardar_informe")

def test_informe():
    d = guardar_informe()
    assert all(k in d for k in CLAVES_INFORME), f"faltan: {[k for k in CLAVES_INFORME if k not in d]}"
    assert json.loads(pathlib.Path("poliglota_unicode.json").read_text())["source"] == source
chequea("R9 informe", test_informe)

## Rúbrica: comprobación automática

In [ ]:
def rubrica():
    for n, e in CHECKS.items():
        print({True: "✅", False: "❌", None: "⏳"}[e], n)
    if DETECTORES and set(DETECTORES) == set(TIPOS):
        try:
            det = evaluar_deteccion({d["id"]: audit_doc(d["text"]) for d in sucios}, verdad, DETECTORES)
            limpios_marcados = sum(any(f(audit_doc(d["text"])) for f in DETECTORES.values()) for d in docs) / len(docs)
            ok = bool((det["recall"] >= 0.95).all() and (det["precisión (vs limpios)"] >= 0.90).all() and limpios_marcados <= 0.02)
            print("✅" if ok else "❌", f"R5 detección: recall mín {det['recall'].min():.2f} (≥ 0.95), precisión mín {det['precisión (vs limpios)'].min():.2f} (≥ 0.90), limpios marcados {limpios_marcados:.1%} (≤ 2 %)")
        except NotImplementedError:
            print("⏳ R5 detección")
    if REPARACION:
        ok = REPARACION["recuperacion"] >= 0.95 and REPARACION["idempotente"] and REPARACION["limpios_intactos"] >= 0.99
        print("✅" if ok else "❌", f"R6 reparación: {REPARACION}")
    if NDCG:
        ok = NDCG["limpio"] - NDCG["sucio"] >= 0.03 and NDCG["limpio"] - NDCG["normalizado"] <= 0.01
        print("✅" if ok else "❌", f"R7 búsqueda: {({k: round(v, 3) for k, v in NDCG.items()})} (caída ≥ 0.03; normalizado a ≤ 0.01 del limpio)")
    if TOKENS:
        print("✅" if TOKENS["ratio"] >= 1.02 else "❌", f"R8 tokens sucio/normalizado = {TOKENS['ratio']:.3f} (≥ 1.02)")
    print("\n(Con corpus REAL los umbrales de R5–R8 no están calibrados: ver la nota de la rúbrica.)" if source == "real" else "")
rubrica()

## Retos extra (nivel experto)
1. **Esqueleto de confusables completo.** Descarga `confusables.txt` de Unicode (UTS #39) y sustituye `CONFUSABLES` por el mapeo oficial *skeleton*; mide el cambio en falsos positivos sobre Wikipedia real.
2. **NFKC selectivo.** Compara `normalizar(nfkc=True)` con tu versión: ¿qué caracteres de tu corpus cambian? Haz una lista de los que destruirían información (`²`, `ﬁ`, `½`).
3. **Detector de *tag characters*** (`U+E0000–E007F`) y de caracteres de la categoría `Cf`: añade ambos a la auditoría y al normalizador, con tests.
4. **Normalización a nivel de grafemas** para truncado seguro de UI: implementa `trunca_grafemas(t, n)` y compárala con `t[:n]` sobre textos con emojis ZWJ.
5. **Mide el efecto sobre un tokenizer real:** tokens/carácter de GPT-2 y de otro encoding (`cl100k_base` si lo puedes descargar) sobre el corpus sucio vs normalizado por idioma.

## Reflexión
* Tu normalizador elimina ZWJ solo entre letras ASCII. ¿Qué rompería eliminarlo siempre? ¿Y qué rompe **no** eliminar el ZWSP en un corpus de entrenamiento?
* ¿Dónde pondrías la normalización en el pipeline de Políglota (ingesta, tokenizer, consulta)? ¿Qué pasa si indexas con una función y consultas con otra?
* ¿Qué decisiones de este módulo condicionan el **vocabulario** del BPE que entrenarás en el módulo 02 (forma de normalización, invisibles, mojibake residual)?

---
# ⛔ SPOILER — intenta resolverlo primero
Debajo está la solución de referencia, completa y ejecutable. Si «Ejecutar todo» llega hasta aquí, estas celdas **sobrescriben** tus funciones con las de referencia.

In [ ]:
# ── SOLUCIÓN · TODO 1 ──
def escritura(ch):
    n = unicodedata.name(ch, "")
    return n.split()[0] if n else "?"

def palabra_mixta(w):
    return len({escritura(c) for c in w if c.isalpha()} & {"LATIN", "CYRILLIC", "GREEK"}) >= 2

In [ ]:
# ── SOLUCIÓN · TODO 2 ──
MOJI_RE = re.compile(r"[\u00c2-\u00f4][\u0080-\u00bf]{1,3}")

def parece_mojibake(t):
    for m in MOJI_RE.finditer(t):
        g = m.group()
        for n in range(len(g), 1, -1):                       # prueba prefijos: ¿decodifican como UTF-8 válido?
            try:
                g[:n].encode("latin-1").decode("utf-8"); return True
            except UnicodeError:
                continue
    return "â€" in t                                        # firma típica de cp1252 (comillas, guiones)

def arregla_mojibake(t):
    for _ in range(3):                                       # el doble mojibake necesita 2 pasadas
        if not parece_mojibake(t):
            break
        for enc in ("latin-1", "cp1252"):
            try:
                nuevo = t.encode(enc).decode("utf-8")
            except UnicodeError:
                continue
            if nuevo != t:
                t = nuevo; break
        else:
            break
    return t

In [ ]:
# ── SOLUCIÓN · TODO 3 ──
ZW_CHARS = set("\u200b\u200c\u200d\u2060\ufeff")
BIDI_CHARS = set("\u202a\u202b\u202c\u202d\u202e\u2066\u2067\u2068\u2069")

def audit_doc(t):
    palabras = regex.findall(r"\w+", t)
    return {"chars": len(t), "graphemes": len(regex.findall(r"\X", t)), "bytes": len(t.encode("utf-8")),
            "no_nfc": t != unicodedata.normalize("NFC", t), "zero_width": sum(c in ZW_CHARS for c in t),
            "bidi": sum(c in BIDI_CHARS for c in t), "nbsp": sum(unicodedata.category(c) == "Zs" and c != " " for c in t),
            "ancho_completo": sum("\uff01" <= c <= "\uff5e" for c in t), "mixed_words": sum(palabra_mixta(w) for w in palabras),
            "mojibake": parece_mojibake(t), "control": sum(unicodedata.category(c) == "Cc" and c not in "\n\t\r" for c in t),
            "fffd": t.count("\ufffd")}

In [ ]:
# ── SOLUCIÓN · TODO 4 ──
ZW_QUITAR = set("\u200b\u2060\ufeff")
CONFUSABLES = {"\u0430": "a", "\u0435": "e", "\u043e": "o", "\u0440": "p", "\u0441": "c", "\u0445": "x", "\u0456": "i",
               "\u03bf": "o", "\u03b1": "a", "\u0455": "s", "\u0458": "j"}

def plegar_ancho(t):
    return "".join(chr(ord(c) - 0xFEE0) if "\uff01" <= c <= "\uff5e" else c for c in t)

def normalizar(t, nfkc=False):
    t = arregla_mojibake(t)
    t = unicodedata.normalize("NFKC" if nfkc else "NFC", t)
    t = plegar_ancho(t)
    t = "".join(c for c in t if c not in ZW_QUITAR and c not in BIDI_CHARS)
    t = re.sub(r"(?<=[A-Za-z0-9])[\u200c\u200d](?=[A-Za-z0-9])", "", t)          # ZWJ/ZWNJ solo se quitan dentro de palabras ASCII
    t = "".join(" " if unicodedata.category(c) == "Zs" else c for c in t)
    t = "".join(c for c in t if not (unicodedata.category(c) == "Cc" and c not in "\n\t\r"))
    return regex.sub(r"\w+", lambda m: "".join(CONFUSABLES.get(c, c) for c in m.group()) if palabra_mixta(m.group()) else m.group(), t)

In [ ]:
# ── SOLUCIÓN · TODO 5 ──
DETECTORES = {"nfd": lambda a: a["no_nfc"], "zero_width": lambda a: a["zero_width"] > 0, "homoglifo": lambda a: a["mixed_words"] > 0,
              "mojibake": lambda a: a["mojibake"], "nbsp": lambda a: a["nbsp"] > 0, "ancho_completo": lambda a: a["ancho_completo"] > 0,
              "bidi": lambda a: a["bidi"] > 0}

audits = {d["id"]: audit_doc(d["text"]) for d in sucios}
det = evaluar_deteccion(audits, verdad, DETECTORES)
display(det.round(3))

In [ ]:
# ── SOLUCIÓN · TODO 6, 7, 8 ──
def calcula_reparacion():
    rec = np.mean([normalizar(d["text"]) == normalizar(orig[d["id"]]["text"]) for d in sucios if d["id"] in verdad])
    idem = all(normalizar(normalizar(d["text"])) == normalizar(d["text"]) for d in sucios)
    intactos = np.mean([normalizar(d["text"]) == unicodedata.normalize("NFC", d["text"]) for d in docs])
    return {"recuperacion": float(rec), "idempotente": bool(idem), "limpios_intactos": float(intactos)}
REPARACION = calcula_reparacion(); print(REPARACION)

In [ ]:
naive = lambda t: re.findall(r"\w+", t.lower())          # tokenizer léxico SIN normalizar: expone el efecto

def ndcg_variante(docs_v):
    dd = [d for d in docs_v if d["lang"] != "code"]; qs = [q for q in queries if q["lang"] != "code"]
    bm = BM25().fit([naive(d["text"]) for d in dd], [d["id"] for d in dd])
    run = {q["qid"]: bm.search(naive(q["text"]), 10) for q in qs}
    return evaluate(run, {q["qid"]: qrels[q["qid"]] for q in qs})["ndcg"]

NDCG = {"limpio": ndcg_variante(docs), "sucio": ndcg_variante(sucios), "normalizado": ndcg_variante([dict(d, text=normalizar(d["text"])) for d in sucios])}
print({k: round(v, 4) for k, v in NDCG.items()})
enc = get_gpt2_encoding()
if enc is not None:
    muestra = [d for d in sucios if d["lang"] != "code"][:150]
    ts = sum(len(enc.encode(d["text"])) for d in muestra); tn = sum(len(enc.encode(normalizar(d["text"]))) for d in muestra)
    TOKENS = {"sucio": ts, "normalizado": tn, "ratio": ts / tn}; print(TOKENS)

def guardar_informe(path="poliglota_unicode.json"):
    bpc = {l: sum(audit_doc(d["text"])["bytes"] for d in docs if d["lang"] == l) / sum(audit_doc(d["text"])["chars"] for d in docs if d["lang"] == l)
           for l in ("es", "en", "code")}
    d = {"source": source, "n_docs": len(docs), "n_contaminados": len(verdad), "deteccion": det.round(4).to_dict(), "reparacion": REPARACION,
         "ndcg": NDCG, "tokens": TOKENS, "bytes_por_char": bpc, "version_unicode": unicodedata.unidata_version}
    pathlib.Path(path).write_text(json.dumps(d, indent=2, ensure_ascii=False, default=float))
    return d
guardar_informe()
for nombre, fn in [("R1 palabra_mixta", test_mixta), ("R2 mojibake", test_mojibake), ("R3 audit_doc", test_audit), ("R4 normalizar", test_normalizar), ("R9 informe", test_informe)]:
    chequea(nombre, fn)
rubrica()

> **📌 Valores de referencia de la solución.** Corpus **sintético**, `FAST_DEV_RUN=True`, semilla 42, `tasa=0.3`: 93 documentos contaminados; recall = precisión = 1.00 en los 7 tipos; recuperación 93/93; nDCG@10 limpio 0.955 / sucio 0.894 / normalizado 0.955; tokens GPT-2 sucio/normalizado 1.049. Con datos **reales** tus números serán distintos (no pudimos medirlos en el entorno de autoría).